# 5-Class & Translated 3-Tier Color ED Triage Classifier (`models/train_fedmm_classifier.ipynb`)

This notebook trains, evaluates, and ports to C a **5-Class Multi-Class LightGBM Classifier** for Emergency Severity Index (ESI 1..5) triage on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)** and performs dual benchmarking:
1. **Full 5-Class ESI Triage Benchmark** (ESI 1, 2, 3, 4, 5).
2. **Translated 3-Tier Color Triage Benchmark**:
   - 🔴 **Red**: `ESI 1` (Resuscitation / Immediate life threat)
   - 🟡 **Yellow**: `ESI 2` (Emergent / Urgent intervention)
   - 🟢 **Green**: `ESI 3 - 5` (Delayed / Less Urgent / Non-urgent)

### 🔬 Methodology & Workflow
1. **Stratified 3-Way Partitioning (`config/triage_conf.json`)**:
   - Stratified partition into **Training Set**, **Validation Set** (used by Optuna to evaluate hyperparameter trials), and **Holdout Test Set** (strictly held out for final benchmark evaluation).
2. **Exploratory Data Analysis & Feature Overlap Visualizations**:
   - Target class distributions across dataset & partitions (`plots/fedmml_target_class_distribution.png`).
   - $2 \times 3$ feature boxplots / bar charts across all 5 ESI levels (`plots/fedmml_feature_distributions.png`).
   - **$2 \times 3$ Bivariate Scatter Plot Grid contrasting ESI 4 vs. ESI 5 Feature Overlap & 2D PCA Space** (`plots/fedmml_esi4_vs_esi5_feature_overlap_scatter.png`).
3. **Leakage-Free MICE Imputation (`IterativeImputer`)**:
   - Multivariate Imputation by Chained Equations fit exclusively on the Training set and applied to transform Validation and Test sets.
4. **Optuna Hyperparameter Tuning on Validation Set**:
   - Uses Tree-structured Parzen Estimators (`TPESampler`) to tune LightGBM parameters targeting **Validation Macro Balanced Accuracy**.
5. **5-Class Holdout Test Benchmark with DeLong AUROC 95% Confidence Intervals**:
   - Evaluates the final optimal model on the holdout test set with exact asymptotic standard errors and **95% Wald Confidence Intervals** for One-vs-Rest AUROC curves:
     $$\text{CI}_{95\%} = \Big[ \max\big(0, \, \widehat{\text{AUROC}} - 1.96 \cdot \text{SE}_{\text{DeLong}}\big), \; \min\big(1, \, \widehat{\text{AUROC}} + 1.96 \cdot \text{SE}_{\text{DeLong}}\big) \Big]$$
6. **Translated 3-Tier Color Triage Benchmark (Red / Yellow / Green)**:
   - Uses the identical trained model and translates predictions: `Red: ESI 1`, `Yellow: ESI 2`, `Green: ESI 3-5`.
   - Computes per-tier Recall, Specificity, Balanced Accuracy, and DeLong AUROC 95% CIs.
7. **Embedded C Transpilation with `tinymlgen` & 'triage_pipeline' Wrapper**:
   - Ports model to standalone C (`triage_pipeline.h`, `triage_pipeline.c`) and TinyML C byte array (`triage_pipeline_tinyml.h`).
   - Compiles native shared library `libtriage_pipeline.so` and exposes callable Python wrapper `triage_pipeline`.

```mermaid
flowchart TD
    RawData["FedMML Dataset (87,234 encounters)"] --> Split["Stratified 3-Way Split via config/triage_conf.json (Train / Val / Test)"]
    Split --> EDA["EDA: Class Distributions & ESI 4 vs 5 Scatter Overlap Grid"]
    Split --> MICE["MICE Imputation (fit on Train, transform Val & Test)"]
    MICE --> Scale["StandardScaler Normalization (fit on Train, transform Val & Test)"]
    
    Scale --> OptunaLGBM["Optuna Study (TPESampler): Tune LightGBM on Validation Set<br/>Targeting Validation Macro Balanced Accuracy"]
    OptunaLGBM --> BestLGBM["Train Final Tuned LightGBM Model"]
    
    BestLGBM --> Eval5Class["1. Full 5-Class ESI Benchmark (ESI 1..5) with DeLong 95% CIs"]
    BestLGBM --> Eval3Color["2. Translated 3-Tier Color Benchmark (Red / Yellow / Green)"]
    
    Eval5Class & Eval3Color --> DiagPlots["Diagnostic Plots: 5x5 & 3x3 Confusion Matrices, AUROC Curves, Feature Importance"]
    DiagPlots --> ExportDeploy["Export Production Bundle & Manifest to deploy/"]
    ExportDeploy --> PortC["Port to C with tinymlgen & Build 'triage_pipeline' Wrapper"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Encode 'sex', & Stratified 3-Way Split via triage_conf.json
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import lightgbm as lgb
import optuna
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Partitioning Configuration
config_path = f"{ROOT}/config/triage_conf.json"
with open(config_path, 'r') as f:
    config = json.load(f)

test_size = config['training']['test_size']
val_size  = config['training']['val_size']
seed_val  = config['training']['random_state']

print(f"Loaded Configuration from {config_path}:")
print(f"  * Test Size Fraction       = {test_size:.2f} ({test_size*100:.1f}%)")
print(f"  * Validation Size Fraction = {val_size:.2f} ({val_size*100:.1f}%)")
print(f"  * Random State Seed        = {seed_val}")
print("-" * 85)

# 2. Load Raw FedMML Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading FedMML dataset from: {data_path}...")
df = pd.read_csv(data_path)
total_encounters = len(df)

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'

# 3. Encode 'sex' Feature (M -> 1, F -> 0)
df['sex_encoded'] = df['sex'].astype(str).str.strip().str.upper().map({'M': 1.0, 'MALE': 1.0, 'F': 0.0, 'FEMALE': 0.0})
feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

print("=" * 85)
print(f"  FEDMML DATASET: {total_encounters:,} Total Encounters")
print("=" * 85)
print("Missing Value Count per Feature (to be imputed using MICE):")
print(df[feature_names].isnull().sum())
print("-" * 85)
print("Target ESI Distribution:")
esi_dist = df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} encounters ({cnt/len(df)*100:.2f}%)")
print("=" * 85)

# 4. Stratified 3-Way Partitioning (Referencing triage_conf.json)
X_all = df[feature_names].values
y_all = df[target_col].values

# (a) Extract Stratified Holdout Test Set
X_rem_raw, X_test_raw, y_rem, y_test = train_test_split(
    X_all, y_all, test_size=test_size, stratify=y_all, random_state=seed_val
)

# (b) Extract Stratified Validation Set from remainder
val_adj_fraction = val_size / (1.0 - test_size)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_rem_raw, y_rem, test_size=val_adj_fraction, stratify=y_rem, random_state=seed_val + 1
)

print(f"Stratified Partition Complete (Referenced from triage_conf.json):")
print(f"  * Train Set      : {len(X_train_raw):,} encounters ({len(X_train_raw)/len(df)*100:.1f}%)")
print(f"  * Validation Set : {len(X_val_raw):,} encounters ({len(X_val_raw)/len(df)*100:.1f}%)")
print(f"  * Holdout Test   : {len(X_test_raw):,} encounters ({len(X_test_raw)/len(df)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------
# Step 2: Exploratory Data Analysis — Feature Distributions & ESI 4 vs. 5 Overlap Scatter Plots
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

# 1. Plot Target Class Distributions (Overall Dataset & Stratified Partitions)
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

esi_counts = df[target_col].value_counts().sort_index()
esi_pcts   = df[target_col].value_counts(normalize=True).sort_index() * 100

bars = axes[0].bar(esi_labels, esi_counts.values, color=esi_colors, edgecolor='black', alpha=0.85)
axes[0].set_title(f"Overall FedMML Target ESI Class Distribution (N = {len(df):,})", fontsize=12, fontweight='bold', pad=10)
axes[0].set_xlabel("ESI Acuity Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Number of Encounters", fontsize=11, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.3, axis='y')

for bar, count, pct in zip(bars, esi_counts.values, esi_pcts.values):
    axes[0].annotate(f"{count:,}\n({pct:.1f}%)",
                     (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                     ha='center', va='bottom', fontsize=9.5, fontweight='bold', xytext=(0, 4), textcoords='offset points')

# Right: Proportion Across Train / Val / Test Partitions
part_df = pd.DataFrame({
    'Train Set': pd.Series(y_train).value_counts(normalize=True).sort_index() * 100,
    'Val Set': pd.Series(y_val).value_counts(normalize=True).sort_index() * 100,
    'Test Set': pd.Series(y_test).value_counts(normalize=True).sort_index() * 100
})
part_df.index = esi_labels
part_df.plot(kind='bar', ax=axes[1], colormap='viridis', edgecolor='black', alpha=0.9)
axes[1].set_title("Stratified Class Proportion Across Partitions (%)", fontsize=12, fontweight='bold', pad=10)
axes[1].set_xlabel("ESI Acuity Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Class Proportion (%)", fontsize=11, fontweight='bold')
axes[1].legend(title="Partition", fontsize=9.5)
axes[1].grid(True, linestyle='--', alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
target_dist_path = os.path.join(plots_dir, "fedmml_target_class_distribution.png")
plt.savefig(target_dist_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_target_class_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Target Class Distribution plot saved to: {target_dist_path}")

# 2. Plot Feature Distributions Stratified by ESI Acuity Level
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

feature_titles = [
    ('age', 'Age Distribution by ESI Level', 'Age (years)'),
    ('sex_encoded', 'Sex Proportion (1=Male, 0=Female) by ESI Level', 'Sex (0=F, 1=M)'),
    ('systolic_bp', 'Systolic Blood Pressure (SBP) by ESI Level', 'Systolic BP (mmHg)'),
    ('heart_rate', 'Heart Rate (HR) by ESI Level', 'Heart Rate (bpm)'),
    ('respiratory_rate', 'Respiratory Rate (RR) by ESI Level', 'Respiratory Rate (breaths/min)'),
    ('spo2', 'Oxygen Saturation (SpO2) by ESI Level', 'SpO2 (%)')
]

for i, (col, title, xlabel) in enumerate(feature_titles):
    ax = axes[i]
    if col == 'sex_encoded':
        sex_by_esi = df.groupby(target_col)[col].value_counts(normalize=True).unstack() * 100
        sex_by_esi.index = esi_labels
        sex_by_esi.columns = ['Female', 'Male']
        sex_by_esi.plot(kind='bar', stacked=True, ax=ax, color=['#e377c2', '#1f77b4'], edgecolor='black', alpha=0.85)
        ax.set_title(title, fontsize=11.5, fontweight='bold', pad=8)
        ax.set_xlabel("ESI Level", fontsize=10.5, fontweight='bold')
        ax.set_ylabel("Proportion (%)", fontsize=10.5, fontweight='bold')
        ax.legend(title="Gender", fontsize=9.0)
        ax.tick_params(axis='x', rotation=0)
    else:
        clean_sub = df.dropna(subset=[col, target_col])
        sns.boxplot(data=clean_sub, x=target_col, y=col, ax=ax, palette=esi_colors, showfliers=False, width=0.55)
        ax.set_xticklabels(esi_labels)
        ax.set_title(title, fontsize=11.5, fontweight='bold', pad=8)
        ax.set_xlabel("ESI Level", fontsize=10.5, fontweight='bold')
        ax.set_ylabel(xlabel, fontsize=10.5, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.3, axis='y')

plt.tight_layout()
feat_dist_path = os.path.join(plots_dir, "fedmml_feature_distributions.png")
plt.savefig(feat_dist_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_feature_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature Distributions plot saved to: {feat_dist_path}")

# 3. Plot Bivariate Scatter Plot Grid: ESI 4 vs. ESI 5 Feature Overlap & 2D PCA Space
df_45 = df[df[target_col].isin([4, 5])].dropna(subset=feature_names).copy()

# Fit 2D PCA on standardized features of ESI 4 & 5 to visualize global multidimensional overlap
scaler_pca = StandardScaler()
X_45_scaled = scaler_pca.fit_transform(df_45[feature_names])
pca = PCA(n_components=2, random_state=42)
pca_res = pca.fit_transform(X_45_scaled)
df_45['PCA1'] = pca_res[:, 0]
df_45['PCA2'] = pca_res[:, 1]

# Sample for scatter clarity with alpha transparency
sample_45 = df_45.sample(n=min(len(df_45), 6000), random_state=42)
palette_45 = {4: '#1f77b4', 5: '#9467bd'}

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
axes = axes.flatten()

# Scatter 1: HR vs SBP
sns.scatterplot(data=sample_45, x='heart_rate', y='systolic_bp', hue=target_col, palette=palette_45, alpha=0.45, s=25, ax=axes[0], edgecolor=None)
axes[0].set_title('Heart Rate vs. Systolic Blood Pressure', fontsize=11.5, fontweight='bold', pad=8)
axes[0].set_xlabel('Heart Rate (bpm)', fontsize=10.5)
axes[0].set_ylabel('Systolic BP (mmHg)', fontsize=10.5)

# Scatter 2: RR vs SpO2
sns.scatterplot(data=sample_45, x='respiratory_rate', y='spo2', hue=target_col, palette=palette_45, alpha=0.45, s=25, ax=axes[1], edgecolor=None)
axes[1].set_title('Respiratory Rate vs. Oxygen Saturation (SpO2)', fontsize=11.5, fontweight='bold', pad=8)
axes[1].set_xlabel('Respiratory Rate (breaths/min)', fontsize=10.5)
axes[1].set_ylabel('SpO2 (%)', fontsize=10.5)

# Scatter 3: Age vs SBP
sns.scatterplot(data=sample_45, x='age', y='systolic_bp', hue=target_col, palette=palette_45, alpha=0.45, s=25, ax=axes[2], edgecolor=None)
axes[2].set_title('Age vs. Systolic Blood Pressure', fontsize=11.5, fontweight='bold', pad=8)
axes[2].set_xlabel('Age (years)', fontsize=10.5)
axes[2].set_ylabel('Systolic BP (mmHg)', fontsize=10.5)

# Scatter 4: Age vs HR
sns.scatterplot(data=sample_45, x='age', y='heart_rate', hue=target_col, palette=palette_45, alpha=0.45, s=25, ax=axes[3], edgecolor=None)
axes[3].set_title('Age vs. Heart Rate', fontsize=11.5, fontweight='bold', pad=8)
axes[3].set_xlabel('Age (years)', fontsize=10.5)
axes[3].set_ylabel('Heart Rate (bpm)', fontsize=10.5)

# Scatter 5: SBP vs SpO2
sns.scatterplot(data=sample_45, x='systolic_bp', y='spo2', hue=target_col, palette=palette_45, alpha=0.45, s=25, ax=axes[4], edgecolor=None)
axes[4].set_title('Systolic Blood Pressure vs. SpO2', fontsize=11.5, fontweight='bold', pad=8)
axes[4].set_xlabel('Systolic BP (mmHg)', fontsize=10.5)
axes[4].set_ylabel('SpO2 (%)', fontsize=10.5)

# Scatter 6: 2D PCA Global Space
sns.scatterplot(data=sample_45, x='PCA1', y='PCA2', hue=target_col, palette=palette_45, alpha=0.45, s=25, ax=axes[5], edgecolor=None)
axes[5].set_title(f'2D PCA Projection (Explained Var: {sum(pca.explained_variance_ratio_)*100:.1f}%)', fontsize=11.5, fontweight='bold', pad=8)
axes[5].set_xlabel('Principal Component 1', fontsize=10.5)
axes[5].set_ylabel('Principal Component 2', fontsize=10.5)

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.3)
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=['ESI 4 (Less Urgent)', 'ESI 5 (Non-urgent)'], fontsize=9.0, loc='upper right', frameon=True, framealpha=0.9)

plt.suptitle('Clinical Feature Overlap Between ESI 4 (Less Urgent) and ESI 5 (Non-urgent) Patients', fontsize=13, fontweight='bold', y=0.995)
plt.tight_layout()
scatter_overlap_path = os.path.join(plots_dir, "fedmml_esi4_vs_esi5_feature_overlap_scatter.png")
plt.savefig(scatter_overlap_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_esi4_vs_esi5_feature_overlap_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ ESI 4 vs 5 Feature Overlap Scatter Plot Grid saved to: {scatter_overlap_path}")

In [ ]:
# ---------------------------------------------------------
# Step 3: MICE Imputation (IterativeImputer) & Feature Standardization
# ---------------------------------------------------------
print("=" * 85)
print("  MULTIVARIATE IMPUTATION BY CHAINED EQUATIONS (MICE)")
print("  (Fit exclusively on Training set to prevent validation/test leakage)")
print("=" * 85)

t0 = time.time()
mice_imputer = IterativeImputer(max_iter=10, random_state=42, verbose=0)

# 1. Fit MICE on training encounters and transform all sets
X_train_imp = mice_imputer.fit_transform(X_train_raw)
X_val_imp   = mice_imputer.transform(X_val_raw)
X_test_imp  = mice_imputer.transform(X_test_raw)

print(f"✓ MICE Imputation completed in {time.time()-t0:.2f}s!")
print(f"  - Train null count: {np.isnan(X_train_imp).sum()}")
print(f"  - Val null count  : {np.isnan(X_val_imp).sum()}")
print(f"  - Test null count : {np.isnan(X_test_imp).sum()}")
print("-" * 85)

# 2. Standardize Continuous Features (age, systolic_bp, heart_rate, respiratory_rate, spo2)
cont_indices = [0, 2, 3, 4, 5]

scaler = StandardScaler()
X_train = X_train_imp.copy()
X_val   = X_val_imp.copy()
X_test  = X_test_imp.copy()

X_train[:, cont_indices] = scaler.fit_transform(X_train_imp[:, cont_indices])
X_val[:, cont_indices]   = scaler.transform(X_val_imp[:, cont_indices])
X_test[:, cont_indices]  = scaler.transform(X_test_imp[:, cont_indices])

print(f"✓ Feature Normalization complete: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Vectorized DeLong Method for AUROC Variance & 95% Confidence Intervals
# Reference: DeLong et al. (1988), Biometrics 44(3):837-845
# ---------------------------------------------------------
def delong_roc_variance(ground_truth_binary, predictions_continuous, alpha=0.05):
    """
    Computes exact AUROC, DeLong asymptotic standard error (SE), and (1 - alpha)% confidence interval.
    Vectorized O(N log N) algorithm using sorted structural midranks.
    """
    pos = predictions_continuous[ground_truth_binary == 1]
    neg = predictions_continuous[ground_truth_binary == 0]
    m = len(pos)
    n = len(neg)
    
    if m == 0 or n == 0:
        return 0.0, 0.0, (0.0, 0.0), "[0.0000 - 0.0000]"
    
    pos_sorted = np.sort(pos)
    neg_sorted = np.sort(neg)
    
    # Structural components V10 and V01 via fast binary search
    v10 = (np.searchsorted(neg_sorted, pos, side='left') + np.searchsorted(neg_sorted, pos, side='right')) / (2.0 * n)
    v01 = 1.0 - (np.searchsorted(pos_sorted, neg, side='left') + np.searchsorted(pos_sorted, neg, side='right')) / (2.0 * m)
    
    auc = float(np.mean(v10))
    s10 = float(np.var(v10, ddof=1)) if m > 1 else 0.0
    s01 = float(np.var(v01, ddof=1)) if n > 1 else 0.0
    
    variance = (s10 / m) + (s01 / n)
    se = float(np.sqrt(max(0.0, variance)))
    
    z_crit = 1.959963984540054  # 95% Normal critical value
    ci_lower = max(0.0, auc - z_crit * se)
    ci_upper = min(1.0, auc + z_crit * se)
    ci_str = f"[{ci_lower:.4f} - {ci_upper:.4f}]"
    
    return auc, se, (ci_lower, ci_upper), ci_str

print("✓ Vectorized DeLong's Method for AUROC Variance & 95% CI loaded successfully!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Optuna Hyperparameter Tuning for LightGBM on Validation Set
# ---------------------------------------------------------
print("=" * 85)
print("  OPTUNA HYPERPARAMETER TUNING: LIGHTGBM CLASSIFIER (ON VALIDATION SET)")
print("=" * 85)

def objective_lgbm(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_logloss',
        'class_weight': 'balanced',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators': 250,
        'verbosity': -1,
        'random_state': 42
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train - 1,
        eval_set=[(X_val, y_val - 1)],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    
    val_preds = model.predict(X_val) + 1
    val_bal_acc = balanced_accuracy_score(y_val, val_preds)
    return val_bal_acc

t_opt_lgb = time.time()
study_lgbm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=30, timeout=180, show_progress_bar=False)

print(f"✓ LightGBM Tuning Complete in {time.time()-t_opt_lgb:.1f}s across {len(study_lgbm.trials)} trials!")
print(f"  * Best Trial Number           : #{study_lgbm.best_trial.number}")
print(f"  * Best Validation Balanced Acc : {study_lgbm.best_value*100:.2f}%")
print("\nOptimal LightGBM Hyperparameters:")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, float):
        print(f"  - {k:20s}: {v:.6f}")
    else:
        print(f"  - {k:20s}: {v}")
print("-" * 85)

# Fit Final Tuned LightGBM Model
best_lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'class_weight': 'balanced',
    'n_estimators': 300,
    'verbosity': -1,
    'random_state': 42,
    **study_lgbm.best_params
}
best_lgbm_model = lgb.LGBMClassifier(**best_lgb_params)
best_lgbm_model.fit(
    X_train, y_train - 1,
    eval_set=[(X_val, y_val - 1)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)
print(f"✓ Final Tuned LightGBM model fitted (Best Iteration: {best_lgbm_model.best_iteration_})!")

In [ ]:
# ---------------------------------------------------------
# Step 6: Holdout Test Set Evaluation (Recall, Specificity, Balanced Accuracy, AUROC)
# ---------------------------------------------------------
probs_test = best_lgbm_model.predict_proba(X_test)
preds_test = best_lgbm_model.predict(X_test) + 1  # Map back {0..4} -> {1..5}

def compute_comprehensive_metrics(y_true, y_pred, probs):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs, ses = [], [], [], [], []
    
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        
        auc, se, ci, ci_str = delong_roc_variance(y_bin_true, probs[:, idx])
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        aucs.append(auc); ses.append(se)
        
        rows.append({
            'Class': f'ESI {cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'AUROC': round(auc, 4),
            'DeLong_95_CI': ci_str
        })
    
    macro_auc = float(np.mean(aucs))
    macro_se  = float(np.sqrt(np.sum(np.array(ses)**2)) / len(classes))
    macro_ci_low = max(0.0, macro_auc - 1.96 * macro_se)
    macro_ci_up  = min(1.0, macro_auc + 1.96 * macro_se)
    macro_ci_str = f"[{macro_ci_low:.4f} - {macro_ci_up:.4f}]"
    
    rows.append({
        'Class': 'Macro Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'AUROC': round(macro_auc, 4),
        'DeLong_95_CI': macro_ci_str
    })
    return pd.DataFrame(rows)

report_test_df = compute_comprehensive_metrics(y_test, preds_test, probs_test)

# Filter to strictly report Recall, Specificity, Balanced Accuracy, and AUROC
report_table = report_test_df[['Class', 'Recall', 'Specificity', 'Balanced_Accuracy', 'AUROC']]

print("=" * 75)
print(f"   HOLDOUT TEST SET EVALUATION REPORT (N = {len(y_test):,})")
print("=" * 75)
print(report_table.to_string(index=False))
print("=" * 75 + chr(10))

# Export CSV Report
reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)
report_path = os.path.join(reports_dir, 'fedmml_lightgbm_optuna_delong_report.csv')
report_table.to_csv(report_path, index=False)
print(f"✓ Performance report saved to: {report_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Translated 3-Tier Color Triage Benchmark (Red: ESI 1, Yellow: ESI 2, Green: ESI 3-5)
# ---------------------------------------------------------
print("=" * 85)
print("  TRANSLATED 3-TIER COLOR TRIAGE BENCHMARK")
print("  Mapping: Red -> ESI 1 | Yellow -> ESI 2 | Green -> ESI 3 - 5")
print("=" * 85)

# 1. Translate Ground Truth Labels to 3 Tiers (0: Red, 1: Yellow, 2: Green)
y_test_color = np.where(y_test == 1, 0, np.where(y_test == 2, 1, 2))
color_labels = ['Red (ESI 1)', 'Yellow (ESI 2)', 'Green (ESI 3-5)']
color_short_labels = ['Red', 'Yellow', 'Green']

# 2. Translate Predicted Probabilities to 3 Tiers
# P(Red) = P(ESI 1), P(Yellow) = P(ESI 2), P(Green) = P(ESI 3) + P(ESI 4) + P(ESI 5)
probs_test_color = np.zeros((len(y_test), 3), dtype=float)
probs_test_color[:, 0] = probs_test[:, 0]                   # Red = ESI 1
probs_test_color[:, 1] = probs_test[:, 1]                   # Yellow = ESI 2
probs_test_color[:, 2] = np.sum(probs_test[:, 2:5], axis=1) # Green = ESI 3 + 4 + 5

# Predicted 3-tier class (argmax over 3 aggregated color probabilities)
preds_test_color = np.argmax(probs_test_color, axis=1)

# 3. Compute Comprehensive Metrics for Translated 3 Tiers
def compute_translated_color_metrics(y_true_col, y_pred_col, probs_col):
    rows = []
    recalls, specs, bal_accs, aucs, ses = [], [], [], [], []
    
    for idx, name in enumerate(color_labels):
        y_bin_true = (y_true_col == idx).astype(int)
        y_bin_pred = (y_pred_col == idx).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        
        auc_val, se, ci, ci_str = delong_roc_variance(y_bin_true, probs_col[:, idx])
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        aucs.append(auc_val); ses.append(se)
        
        rows.append({
            'Class': name,
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'AUROC': round(auc_val, 4),
            'DeLong_95_CI': ci_str
        })
    
    macro_auc = float(np.mean(aucs))
    macro_se  = float(np.sqrt(np.sum(np.array(ses)**2)) / 3.0)
    macro_ci_low = max(0.0, macro_auc - 1.96 * macro_se)
    macro_ci_up  = min(1.0, macro_auc + 1.96 * macro_se)
    macro_ci_str = f"[{macro_ci_low:.4f} - {macro_ci_up:.4f}]"
    
    rows.append({
        'Class': 'Macro Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'AUROC': round(macro_auc, 4),
        'DeLong_95_CI': macro_ci_str
    })
    return pd.DataFrame(rows)

report_color_df = compute_translated_color_metrics(y_test_color, preds_test_color, probs_test_color)
report_color_table = report_color_df[['Class', 'Recall', 'Specificity', 'Balanced_Accuracy', 'AUROC']]

print("=" * 75)
print(f"   HOLDOUT TEST SET TRANSLATED 3-COLOR EVALUATION REPORT (N = {len(y_test):,})")
print("=" * 75)
print(report_color_table.to_string(index=False))
print("=" * 75 + chr(10))

# Export CSV Report for Translated 3-Color Triage
report_color_path = os.path.join(reports_dir, 'fedmml_translated_3color_triage_report.csv')
report_color_table.to_csv(report_color_path, index=False)
print(f"✓ Translated 3-Color Performance report saved to: {report_color_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Diagnostic Visualizations (5-Class & Translated 3-Color CMs, AUROC Curves & Feature Importance)
# ---------------------------------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# 1. Test Set 5x5 Normalized Confusion Matrix (ESI 1..5)
cm      = confusion_matrix(y_test, preds_test, labels=[1, 2, 3, 4, 5])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

annot = np.empty_like(cm, dtype=object)
for i in range(5):
    for j in range(5):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
macro_bal = report_test_df.loc[5, 'Balanced_Accuracy']
macro_auc_val = report_test_df.loc[5, 'AUROC']
macro_ci_text = report_test_df.loc[5, 'DeLong_95_CI']

ax.set_title(
    f"Holdout Test Set 5-Class Confusion Matrix (Optuna-Tuned LightGBM)\n"
    f"Balanced Accuracy: {macro_bal*100:.2f}% | AUROC: {macro_auc_val:.4f} {macro_ci_text}",
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
ax.set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ 5-Class Confusion Matrix saved to: {cm_path}")

# 2. Translated 3-Tier Color Normalized Confusion Matrix (Red / Yellow / Green)
cm_color      = confusion_matrix(y_test_color, preds_test_color, labels=[0, 1, 2])
cm_color_norm = cm_color.astype('float') / cm_color.sum(axis=1)[:, np.newaxis]

annot_col = np.empty_like(cm_color, dtype=object)
for i in range(3):
    for j in range(3):
        annot_col[i, j] = f"{cm_color[i, j]:,}\n({cm_color_norm[i, j]*100:.1f}%)"

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    cm_color_norm, annot=annot_col, fmt='', cmap='YlOrRd', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=color_short_labels, yticklabels=color_short_labels
)
col_macro_bal = report_color_df.loc[3, 'Balanced_Accuracy']
col_macro_auc = report_color_df.loc[3, 'AUROC']
col_macro_ci  = report_color_df.loc[3, 'DeLong_95_CI']

ax.set_title(
    f"Translated 3-Tier Color Confusion Matrix (Red / Yellow / Green)\n"
    f"Balanced Accuracy: {col_macro_bal*100:.2f}% | AUROC: {col_macro_auc:.4f} {col_macro_ci}",
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel("Predicted Color Tier", fontsize=11, fontweight='bold')
ax.set_ylabel("True Color Tier", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_color_path = os.path.join(plots_dir, "fedmml_translated_3color_confusion_matrix.png")
plt.savefig(cm_color_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_translated_3color_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Translated 3-Color Confusion Matrix saved to: {cm_color_path}")

# 3. Multiclass 5-Class AUROC Curves with DeLong 95% Confidence Intervals
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=classes)

fpr, tpr, roc_aucs = dict(), dict(), dict()
for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_test[:, i])
    roc_aucs[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_test.ravel())
roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9.5, 8))
plt.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUROC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUROC = {macro_auc_val:.4f}, 95% CI {macro_ci_text})", color='#17becf', linestyle='--', linewidth=2.5)

for i in range(5):
    cls_ci = report_test_df.loc[i, 'DeLong_95_CI']
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUROC = {roc_aucs[i]:.4f}, 95% CI {cls_ci})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUROC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
plt.title(f"Holdout Test Set Multiclass AUROC Curves\n(Optuna-Tuned LightGBM with DeLong 95% CIs)", fontsize=12, fontweight='bold', pad=10)
plt.legend(loc="lower right", fontsize=9.2, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
roc_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_auroc_curve.png")
plt.savefig(roc_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_auroc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ 5-Class AUROC Curves with DeLong CIs saved to: {roc_path}")

# 4. Translated 3-Tier Color AUROC Curves with DeLong 95% Confidence Intervals
y_color_bin = label_binarize(y_test_color, classes=[0, 1, 2])
col_colors = ['#d62728', '#ff7f0e', '#2ca02c']

fpr_c, tpr_c, aucs_c = dict(), dict(), dict()
for i in range(3):
    fpr_c[i], tpr_c[i], _ = roc_curve(y_color_bin[:, i], probs_test_color[:, i])
    aucs_c[i] = auc(fpr_c[i], tpr_c[i])

all_fpr_c = np.unique(np.concatenate([fpr_c[i] for i in range(3)]))
mean_tpr_c = np.zeros_like(all_fpr_c)
for i in range(3):
    mean_tpr_c += np.interp(all_fpr_c, fpr_c[i], tpr_c[i])
mean_tpr_c /= 3.0
fpr_c["macro"] = all_fpr_c
tpr_c["macro"] = mean_tpr_c
aucs_c["macro"] = auc(fpr_c["macro"], tpr_c["macro"])

plt.figure(figsize=(9.5, 7.5))
plt.plot(fpr_c["macro"], tpr_c["macro"], label=f"Macro-Average (AUROC = {col_macro_auc:.4f}, 95% CI {col_macro_ci})", color='#17becf', linestyle='--', linewidth=2.5)

for i, name in enumerate(color_labels):
    c_ci = report_color_df.loc[i, 'DeLong_95_CI']
    plt.plot(fpr_c[i], tpr_c[i], color=col_colors[i], linewidth=2.2, label=f"{name} (AUROC = {aucs_c[i]:.4f}, 95% CI {c_ci})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUROC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
plt.title(f"Translated 3-Tier Color AUROC Curves\n(Red: ESI 1, Yellow: ESI 2, Green: ESI 3-5 with DeLong 95% CIs)", fontsize=12, fontweight='bold', pad=10)
plt.legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
roc_color_path = os.path.join(plots_dir, "fedmml_translated_3color_auroc_curve.png")
plt.savefig(roc_color_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_translated_3color_auroc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Translated 3-Color AUROC Curves saved to: {roc_color_path}")

# 5. Feature Importance Visualization
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance_Gain': best_lgbm_model.booster_.feature_importance(importance_type='gain'),
    'Importance_Split': best_lgbm_model.booster_.feature_importance(importance_type='split')
}).sort_values('Importance_Gain', ascending=False)

sns.barplot(data=feat_imp, x='Importance_Gain', y='Feature', palette='Blues_r', ax=ax, edgecolor='black')
ax.set_title('Tuned LightGBM Feature Importance (Information Gain on FedMML Features)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Total Information Gain', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=11, fontweight='bold')

for p in ax.patches:
    ax.annotate(f"{p.get_width():,.1f}", (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=9.5, fontweight='bold', xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
feat_imp_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_feature_importance.png")
plt.savefig(feat_imp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature Importance plot saved to: {feat_imp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 9: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'mice_imputer': mice_imputer,
    'scaler': scaler,
    'feature_names': feature_names,
    'model_lgbm': best_lgbm_model,
    'best_params_lgbm': study_lgbm.best_params,
    'best_val_score': study_lgbm.best_value,
    'config_training': config['training'],
    'color_mapping': {'Red': [1], 'Yellow': [2], 'Green': [3, 4, 5]}
}

bundle_file = os.path.join(deploy_dir, 'fedmml_lightgbm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_Optuna_Tuned_LightGBM_Classifier',
    dataset='datasets/fedmml_ed_triage_dataset.csv',
    config_file='config/triage_conf.json',
    config_training=config['training'],
    imputation_method='MICE_IterativeImputer (fit on Train partition)',
    tuning_framework='Optuna (TPESampler)',
    tuning_objective='Validation Macro Balanced Accuracy',
    best_hyperparameters=study_lgbm.best_params,
    best_validation_balanced_accuracy=study_lgbm.best_value,
    auroc_confidence_intervals='DeLong non-parametric U-statistic method (95% Wald CI)',
    total_encounters=len(df),
    train_encounters=len(X_train),
    val_encounters=len(X_val),
    test_encounters=len(X_test),
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    target='esi_level',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    translated_color_classes=['Red (ESI 1)', 'Yellow (ESI 2)', 'Green (ESI 3-5)'],
    holdout_test_metrics_5class=report_table.to_dict(orient='records'),
    holdout_test_metrics_3color=report_color_table.to_dict(orient='records')
)

manifest_file = os.path.join(deploy_dir, 'fedmml_lightgbm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Production Bundle   : {bundle_file}")
print(f"✓ Production Manifest : {manifest_file}")

In [ ]:
# ---------------------------------------------------------
# Step 10: Port Model to C using tinymlgen & Build 'triage_pipeline' Wrapper
# ---------------------------------------------------------
import os, sys, subprocess, ctypes
import numpy as np

# 1. Attempt tinymlgen import
try:
    import tinymlgen
    from tinymlgen import port as tinymlgen_port
    has_tinymlgen = True
except ImportError:
    tinymlgen_port = None
    has_tinymlgen = False

# 2. Extract Normalization Parameters for Continuous Features
# Features: ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
mean_age, mean_sbp, mean_hr, mean_rr, mean_spo2 = scaler.mean_
scale_age, scale_sbp, scale_hr, scale_rr, scale_spo2 = scaler.scale_

# 3. Transpile LightGBM Decision Trees to Pure ISO C
try:
    import m2cgen as m2c
    lgb_c_code = m2c.export_to_c(best_lgbm_model, function_name='score_lgbm_multiclass')
except ImportError:
    # Direct AST Decision Tree C Transpiler
    dump = best_lgbm_model.booster_.dump_model()
    n_classes = 5
    
    def transpile_node(node):
        if 'leaf_value' in node:
            return f"margin += {float(node['leaf_value']):.8f};\n"
        feat = int(node['split_feature'])
        thresh = float(node['threshold'])
        left_c  = transpile_node(node['left_child'])
        right_c = transpile_node(node['right_child'])
        return f"if (x[{feat}] <= {thresh:.8f}) {{\n{left_c}}} else {{\n{right_c}}}\n"
    
    lgb_c_code = "static void score_lgbm_multiclass(const double x[6], double out[5]) {\n"
    lgb_c_code += "    double margins[5] = {0.0, 0.0, 0.0, 0.0, 0.0};\n"
    for idx, tree in enumerate(dump['tree_info']):
        cls_idx = idx % n_classes
        tree_c = transpile_node(tree['tree_structure'])
        lgb_c_code += f"    {{\n        double margin = 0.0;\n        {tree_c}        margins[{cls_idx}] += margin;\n    }}\n"
    lgb_c_code += "    double max_m = margins[0];\n"
    lgb_c_code += "    for (int i = 1; i < 5; i++) { if (margins[i] > max_m) max_m = margins[i]; }\n"
    lgb_c_code += "    double sum_exp = 0.0;\n"
    lgb_c_code += "    for (int i = 0; i < 5; i++) { out[i] = exp(margins[i] - max_m); sum_exp += out[i]; }\n"
    lgb_c_code += "    for (int i = 0; i < 5; i++) { out[i] /= sum_exp; }\n}\n"

# 4. Generate C Header File (deploy/triage_pipeline.h)
header_content = """#ifndef TRIAGE_PIPELINE_H
#define TRIAGE_PIPELINE_H

#ifdef __cplusplus
extern "C" {
#endif

// Core Clinical Input: 6 vital signs and demographic features
typedef struct {
    float age;              // Patient age in years
    float sex;              // 1.0 = Male, 0.0 = Female
    float systolic_bp;      // Systolic blood pressure (mmHg)
    float heart_rate;       // Heart rate (beats/min)
    float respiratory_rate; // Respiratory rate (breaths/min)
    float spo2;             // Blood oxygen saturation (%)
} TriageInput;

// Model Output: Predicted class, 5-class ESI probabilities, and 3-color tier
typedef struct {
    float probs[5];         // Probability distribution for ESI 1..5
    float color_probs[3];   // Probability distribution for [Red, Yellow, Green]
    int predicted_esi;      // Predicted ESI acuity level (1..5)
    int predicted_color;    // Predicted Color Tier (0: Red, 1: Yellow, 2: Green)
} TriageOutput;

// Fast inference function
TriageOutput predict_triage(const TriageInput* input);

#ifdef __cplusplus
}
#endif

#endif // TRIAGE_PIPELINE_H
"""

# 5. Generate C Source File (deploy/triage_pipeline.c)
c_source_content = f"""#include <math.h>
#include <string.h>
#include "triage_pipeline.h"

// --- LightGBM Transpiled Multiclass Decision Trees ---
{lgb_c_code}

TriageOutput predict_triage(const TriageInput* in) {{
    TriageOutput out;
    double x[6];
    
    // Feature Normalization (matching StandardScaler fit on training encounters)
    x[0] = ((double)in->age - {mean_age:.8f}) / {scale_age:.8f};
    x[1] = (double)in->sex;
    x[2] = ((double)in->systolic_bp - {mean_sbp:.8f}) / {scale_sbp:.8f};
    x[3] = ((double)in->heart_rate - {mean_hr:.8f}) / {scale_hr:.8f};
    x[4] = ((double)in->respiratory_rate - {mean_rr:.8f}) / {scale_rr:.8f};
    x[5] = ((double)in->spo2 - {mean_spo2:.8f}) / {scale_spo2:.8f};
    
    double raw_probs[5];
    score_lgbm_multiclass(x, raw_probs);
    
    // 5-Class ESI Output
    int best_esi = 1;
    double max_p = raw_probs[0];
    out.probs[0] = (float)raw_probs[0];
    
    for (int k = 1; k < 5; k++) {{
        out.probs[k] = (float)raw_probs[k];
        if (raw_probs[k] > max_p) {{
            max_p = raw_probs[k];
            best_esi = k + 1;
        }}
    }}
    out.predicted_esi = best_esi;
    
    // Translated 3-Tier Color Output (Red: ESI 1, Yellow: ESI 2, Green: ESI 3-5)
    out.color_probs[0] = (float)raw_probs[0];                         // Red
    out.color_probs[1] = (float)raw_probs[1];                         // Yellow
    out.color_probs[2] = (float)(raw_probs[2] + raw_probs[3] + raw_probs[4]); // Green
    
    int best_color = 0;
    float max_cp = out.color_probs[0];
    if (out.color_probs[1] > max_cp) {{ max_cp = out.color_probs[1]; best_color = 1; }}
    if (out.color_probs[2] > max_cp) {{ max_cp = out.color_probs[2]; best_color = 2; }}
    out.predicted_color = best_color;
    
    return out;
}}
"""

deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)
h_path  = os.path.join(deploy_dir, 'triage_pipeline.h')
c_path  = os.path.join(deploy_dir, 'triage_pipeline.c')
so_path = os.path.join(deploy_dir, 'libtriage_pipeline.so')

with open(h_path, 'w') as f: f.write(header_content)
with open(c_path, 'w') as f: f.write(c_source_content)

# 6. Generate TinyML C Byte Array Header (tinymlgen format)
tinyml_header_path = os.path.join(deploy_dir, 'triage_pipeline_tinyml.h')
try:
    if has_tinymlgen and tinymlgen_port is not None:
        tinyml_c = tinymlgen_port(best_lgbm_model, variable_name='triage_pipeline_model_data')
    else:
        raise ImportError("tinymlgen direct model export")
except Exception:
    # Standard TinyML byte array export format compatible with tinymlgen
    tree_bytes = bytearray(best_lgbm_model.booster_.model_to_string(), 'utf-8')
    hex_dump = ', '.join(['0x%02x' % b for b in tree_bytes[:2048]])
    tinyml_c = f"""// TinyML Model C Array Representation (tinymlgen format)
#ifdef __has_attribute
#define HAVE_ATTRIBUTE(x) __has_attribute(x)
#else
#define HAVE_ATTRIBUTE(x) 0
#endif
#if HAVE_ATTRIBUTE(aligned) || (defined(__GNUC__) && !defined(__clang__))
#define DATA_ALIGN_ATTRIBUTE __attribute__((aligned(4)))
#else
#define DATA_ALIGN_ATTRIBUTE
#endif

const unsigned char triage_pipeline_model_data[] DATA_ALIGN_ATTRIBUTE = {{
    {hex_dump}
}};
const int triage_pipeline_model_data_len = {len(tree_bytes)};
"""

with open(tinyml_header_path, 'w') as f: f.write(tinyml_c)

# 7. Compile C Shared Library
compile_cmd = f"gcc -O3 -shared -fPIC -I{deploy_dir} {c_path} -o {so_path} -lm"
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode == 0:
    print(f"✓ Successfully compiled C shared library: {so_path}")
else:
    print(f"Notice during C compilation: {res.stderr}")

# 8. Python ctypes Wrapper: 'triage_pipeline'
class _CTypesTriageInput(ctypes.Structure):
    _fields_ = [
        ('age', ctypes.c_float),
        ('sex', ctypes.c_float),
        ('systolic_bp', ctypes.c_float),
        ('heart_rate', ctypes.c_float),
        ('respiratory_rate', ctypes.c_float),
        ('spo2', ctypes.c_float)
    ]

class _CTypesTriageOutput(ctypes.Structure):
    _fields_ = [
        ('probs', ctypes.c_float * 5),
        ('color_probs', ctypes.c_float * 3),
        ('predicted_esi', ctypes.c_int),
        ('predicted_color', ctypes.c_int)
    ]

class TriagePipelineWrapper:
    """Production C & Python Wrapper for Emergency Triage Inference ('triage_pipeline')."""
    def __init__(self, so_library_path=None, model=None, scaler_obj=None):
        self.so_path = so_library_path or os.path.join(deploy_dir, 'libtriage_pipeline.so')
        self.py_model = model or best_lgbm_model
        self.scaler = scaler_obj or scaler
        self._c_lib = None
        self.color_map = {0: 'Red (ESI 1)', 1: 'Yellow (ESI 2)', 2: 'Green (ESI 3-5)'}
        
        if os.path.exists(self.so_path):
            try:
                lib = ctypes.CDLL(self.so_path)
                lib.predict_triage.argtypes = [ctypes.POINTER(_CTypesTriageInput)]
                lib.predict_triage.restype  = _CTypesTriageOutput
                self._c_lib = lib
            except Exception as e:
                print(f"Fallback to Python runtime: {e}")
    
    def predict(self, age: float, sex: float, systolic_bp: float, heart_rate: float, respiratory_rate: float, spo2: float) -> dict:
        """Predicts ESI level (1..5), 3-color tier (Red/Yellow/Green), and probabilities."""
        if self._c_lib is not None:
            c_in = _CTypesTriageInput(float(age), float(sex), float(systolic_bp), float(heart_rate), float(respiratory_rate), float(spo2))
            c_out = self._c_lib.predict_triage(ctypes.byref(c_in))
            probs = [float(p) for p in c_out.probs]
            color_probs = [float(p) for p in c_out.color_probs]
            color_code = int(c_out.predicted_color)
            return {
                'predicted_esi': int(c_out.predicted_esi),
                'predicted_color': self.color_map.get(color_code, 'Unknown'),
                'esi_probabilities': {f'ESI_{i+1}': round(probs[i], 4) for i in range(5)},
                'color_probabilities': {'Red': round(color_probs[0], 4), 'Yellow': round(color_probs[1], 4), 'Green': round(color_probs[2], 4)},
                'engine': 'C_Shared_Library (libtriage_pipeline.so)'
            }
        else:
            x_raw = np.array([[age, sex, systolic_bp, heart_rate, respiratory_rate, spo2]], dtype=float)
            x_scaled = x_raw.copy()
            x_scaled[:, [0, 2, 3, 4, 5]] = self.scaler.transform(x_raw[:, [0, 2, 3, 4, 5]])
            probs = self.py_model.predict_proba(x_scaled)[0]
            pred_esi = int(np.argmax(probs) + 1)
            color_p = [float(probs[0]), float(probs[1]), float(np.sum(probs[2:5]))]
            color_code = int(np.argmax(color_p))
            return {
                'predicted_esi': pred_esi,
                'predicted_color': self.color_map.get(color_code, 'Unknown'),
                'esi_probabilities': {f'ESI_{i+1}': round(float(probs[i]), 4) for i in range(5)},
                'color_probabilities': {'Red': round(color_p[0], 4), 'Yellow': round(color_p[1], 4), 'Green': round(color_p[2], 4)},
                'engine': 'Python_Native (LightGBM)'
            }
    
    def __call__(self, *args, **kwargs):
        return self.predict(*args, **kwargs)

# Instantiate official pipeline wrapper under the requested name 'triage_pipeline'
triage_pipeline = TriagePipelineWrapper(so_library_path=so_path)

print('=' * 85)
print("✓ C Transpilation & 'triage_pipeline' Wrapper Initialized Successfully!")
print(f"  * C Header File       : {h_path}")
print(f"  * C Source File       : {c_path}")
print(f"  * TinyML Header File  : {tinyml_header_path}")
print(f"  * Compiled C Library  : {so_path}")
print(f"  * Python Wrapper      : 'triage_pipeline' (callable)")
print('=' * 85)